In [14]:
import pandas as pd
product_meta = pd.read_csv("../12_22/final_product_with_brandtone_meta.csv")
print(list(product_meta.columns))

['brand', '상품명', 'category', 'subcategory']


In [1]:
import numpy as np
import pandas as pd
import re

# ==============================
# PATH
# ==============================
PERSONA_NPY  = "../data_csv/persona_vectors.npy"
PERSONA_META = "../data_csv/persona_meta.csv"

PRODUCT_NPY  = "../data_csv/final_product_with_brandtone.npy"
PRODUCT_META = "../data_csv/final_product_with_brandtone_meta.csv"

AMORE_FINAL  = "../data_csv/amore_final.csv"
OUT_CSV      = "../data_csv/persona_product_similarity_full.csv"

EPS = 1e-8

# ==============================
# LOAD
# ==============================
P = np.load(PERSONA_NPY).astype(np.float32)      # (P, D)
X = np.load(PRODUCT_NPY).astype(np.float32)      # (N, D)

persona_meta = pd.read_csv(PERSONA_META)
product_meta = pd.read_csv(PRODUCT_META)
amore_df     = pd.read_csv(AMORE_FINAL)

assert P.shape[0] == len(persona_meta)
assert X.shape[0] == len(product_meta)

# ==============================
# TONE SPLIT
# ==============================
RISK_DIM = 4
persona_dim = P.shape[1]
tone_dim = (persona_dim - RISK_DIM) // 2

p_tone = P[:, :tone_dim]
x_brandtone = X[:, -tone_dim:]

def normalize(v):
    return v / (np.linalg.norm(v, axis=1, keepdims=True) + EPS)

sim_brand = normalize(p_tone) @ normalize(x_brandtone).T   # (P, N)

# ==============================
# META JOIN (ROW ID 기준)
# ==============================
product_meta = product_meta.reset_index(drop=True)
product_meta["__row_id"] = product_meta.index   # 🔴 X 기준축

for df in [product_meta, amore_df]:
    df["brand"] = df["brand"].astype(str)
    df["상품명"] = df["상품명"].astype(str)

product_meta = product_meta.merge(
    amore_df[["brand", "상품명", "전성분"]],
    on=["brand", "상품명"],
    how="left"
)

# ==============================
# ING NORMALIZE + GLOB
# ==============================
def norm(s):
    s = str(s).lower()
    s = re.sub(r"[\*\#\(\)\[\]\{\}\/\:\.]", " ", s)
    s = re.sub(r"[^0-9a-z가-힣\s\-]", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

product_meta["_ing_norm"] = product_meta["전성분"].fillna("").map(norm)

PERSONA_GLOB = {
    "persona_1": ["하이알루로","히알루로","hyalur","hyaluron","시카","centella","asiatic","madecass","병풀"],
    "persona_2": ["나이아신","niacin","niacinamide"],
    "persona_3": ["병풀","centella","asiatica","인삼","ginseng","panax"],
}

def glob_score(patterns, text):
    hits = sum(1 for p in patterns if p in text)
    return 0.0 if hits == 0 else min(1.0, hits / len(patterns))

persona_weights = {
    "persona_1": dict(brand=0.6, ing=0.4),
    "persona_2": dict(brand=0.5, ing=0.5),
    "persona_3": dict(brand=0.4, ing=0.6),
}

# ==============================
# FULL SIMILARITY (X 기준으로 맞춤)
# ==============================
rows = []
product_cols = [c for c in product_meta.columns if not c.startswith("_")]
N = X.shape[0]

for p_idx in range(P.shape[0]):
    pid = persona_meta.loc[p_idx, "persona_id"]
    patterns = [norm(p) for p in PERSONA_GLOB.get(pid, [])]
    w = persona_weights.get(pid, dict(brand=0.6, ing=0.4))

    # 🔴 X 기준 ingredient similarity
    sim_ing_full = np.zeros(N, dtype=np.float32)

    tmp_scores = product_meta["_ing_norm"].apply(
        lambda t: glob_score(patterns, t)
    ).values.astype(np.float32)

    sim_ing_full[product_meta["__row_id"].values] = tmp_scores

    sim = (w["brand"] * sim_brand[p_idx]) + (w["ing"] * sim_ing_full)

    for prod_idx in range(N):
        row = {
            "persona_id": pid,
            "similarity": float(sim[prod_idx]),
            "sim_brand": float(sim_brand[p_idx, prod_idx]),
            "sim_ingredient": float(sim_ing_full[prod_idx]),
            "product_index": int(prod_idx),
        }
        for c in product_cols:
            row[c] = product_meta.loc[prod_idx, c]
        rows.append(row)

# ==============================
# SAVE
# ==============================
result_df = pd.DataFrame(rows)
result_df.to_csv(OUT_CSV, index=False)

print("✅ saved:", OUT_CSV)
print("rows:", len(result_df))

KeyboardInterrupt: 

In [6]:
import pandas as pd

ingredient_meta = pd.read_csv("../12_22/ingredient_meta.csv")

# 1) 전체 성분 수
print("총 성분 수:", len(ingredient_meta))

# 2) 상위 50개 성분 샘플
ingredient_meta["ingredient_name"].head(50)

총 성분 수: 3149


0                                  #1. 부메랑 칼슘티타늄보로실리케이트
1                                          #2. 피치파우트 탤크
2                                               #핑키 마이카
3                                              (1제) 정제수
4                                          (STEP 1) 정제수
5                            (로지) 하이드로제네이티드폴리(C6-14올레핀)
6                                             (소프너) 정제수
7                         (스카이코랄) 하이드로제네이티드폴리(C6-14올레핀)
8                  *BLACK TEA PEPTIDE ACTIVATORTM / 정제수
9                                       *맨 리차징 토너 : 정제수
10                                   *소듐아세틸레이티드하이알루로네이트
11                                          *소듐하이알루로네이트
12                                    *소듐하이알루로네이트크로스폴리머
13                                         *포타슘하이알루로네이트
14                               *하이드록시프로필트라이모늄하이알루로네이트
15                                  *하이드롤라이즈드소듐하이알루로네이트
16                                   *하이드롤라이즈드하이알루로닉애씨드
17                                           *하이

In [7]:
# 3) 특정 키워드 포함 성분이 실제로 있는지
keywords = ["centella", "asiatica", "hyal", "niacin", "ginseng"]

for kw in keywords:
    matched = ingredient_meta[
        ingredient_meta["ingredient_name"].str.lower().str.contains(kw, na=False)
    ]
    print(f"\n[{kw}] 매칭 수:", len(matched))
    print(matched["ingredient_name"].head(10).tolist())


[centella] 매칭 수: 0
[]

[asiatica] 매칭 수: 0
[]

[hyal] 매칭 수: 0
[]

[niacin] 매칭 수: 0
[]

[ginseng] 매칭 수: 0
[]
